# Australian Wage Growth vs Inflation
Have wages kept up with the cost of living? Compare the ABS Wage Price Index (WPI) with the Consumer Price Index (CPI), quarterly, 1997 to June 2026.

**Data (ABS, unchanged downloads in `data/`):**
- CPI, Australia, Table 17: Quarterly All Groups. Series `A2325846C` (All groups CPI, Australia = weighted average of eight capital cities, index numbers, original).
- WPI, Australia, Table 1: total hourly rates of pay excluding bonuses, all industries, original index. Private and Public `A2603609J`, Private `A2603039T`, Public `A2603989W`.

**Method:** annual (year-on-year) growth avoids seasonality. Real wage growth = (1 + WPI growth) / (1 + CPI growth) - 1. The real wage index = WPI / CPI.
**Caveat:** the WPI measures pay for a fixed job, the CPI prices a basket of goods, so "real wages" here is an approximation, and it ignores tax, hours and household composition.

In [ ]:
import json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
CPI_FILE, WPI_FILE = "../data/6401017.xlsx", "../data/634501.xlsx"

def load(path, ids):
    raw = pd.read_excel(path, sheet_name="Data1", header=None)
    r = raw.index[raw[0] == "Series ID"][0]
    out = {}
    for name, sid in ids.items():
        col = raw.columns[raw.iloc[r] == sid][0]
        out[name] = pd.Series(raw.iloc[r + 1:, col].astype(float).values,
                              index=pd.to_datetime(raw.iloc[r + 1:, 0]).dt.to_period("Q"))
    return pd.DataFrame(out)

cpi = load(CPI_FILE, {"cpi": "A2325846C"})
wpi = load(WPI_FILE, {"wpi": "A2603609J", "wpi_private": "A2603039T", "wpi_public": "A2603989W", "wpi_yoy_abs": "A2603611V"})
df = cpi.join(wpi, how="inner")
print("Aligned quarters:", len(df), "|", df.index.min(), "to", df.index.max(), "| missing:", int(df.isna().sum().sum()))
print(df.tail(3).round(2).to_string())

## 1. Compute growth and validate against ABS-published figures

In [ ]:
df["cpi_yoy"] = df.cpi.pct_change(4) * 100
df["wpi_yoy"] = df.wpi.pct_change(4) * 100
df["real_yoy"] = ((1 + df.wpi_yoy / 100) / (1 + df.cpi_yoy / 100) - 1) * 100
df["real_idx"] = df.wpi / df.cpi * 100
for k in ("private", "public"):
    df[f"real_{k}"] = df[f"wpi_{k}"] / df.cpi * 100

# Check 1: my WPI annual change vs the ABS-published WPI annual change series
d = (df.wpi_yoy - df.wpi_yoy_abs).abs().dropna()
print("WPI annual change vs ABS published: max difference %.3f pp over %d quarters" % (d.max(), len(d)))
assert d.max() < 0.1
# Check 2: my CPI annual change vs annual CPI figures printed on the ABS CPI release page (August 2026)
published = {"2022Q3": 7.3, "2022Q4": 7.9, "2023Q1": 7.0, "2023Q2": 6.0, "2023Q3": 5.3, "2023Q4": 4.1,
             "2024Q1": 3.6, "2024Q2": 3.8, "2024Q3": 2.9, "2024Q4": 2.4, "2025Q1": 2.4}
chk = pd.DataFrame({"ABS_published": pd.Series(published), "computed": df.cpi_yoy.loc[list(published)].round(2).values}, index=list(published))
print(chk.to_string()); assert (chk.ABS_published - chk.computed).abs().max() < 0.1
print("Both checks passed.")

## 2. When did real wages fall, and how far?

In [ ]:
neg = df.real_yoy < 0
grp = (neg != neg.shift()).cumsum()
eps = []
for g, blk in df[neg].groupby(grp[neg]):
    eps.append({"start": str(blk.index[0]), "end": str(blk.index[-1]), "quarters": len(blk),
                "worst_real_wage_growth_%": round(blk.real_yoy.min(), 2), "worst_quarter": str(blk.real_yoy.idxmin())})
eps = pd.DataFrame(eps); print(eps.to_string(index=False)); eps.to_csv("../outputs/real_wage_episodes.csv", index=False)

peak_q = df.real_idx.loc["2019Q4":"2022Q4"].idxmax()
after = df.real_idx.loc[peak_q:]
trough_q = after.idxmin()
drop = (after.min() / df.real_idx[peak_q] - 1) * 100
rec = after.loc[trough_q:][after.loc[trough_q:] >= df.real_idx[peak_q]]
latest_q = df.index[-1]
vs_peak = (df.real_idx[latest_q] / df.real_idx[peak_q] - 1) * 100
print("\nReal wage index (WPI/CPI) peak before the surge:", peak_q, "| trough:", trough_q,
      "| peak-to-trough: %.1f%%" % drop)
print("Recovered to the old peak:", str(rec.index[0]) if len(rec) else "not yet", "| latest vs peak: %+.1f%% (%s)" % (vs_peak, latest_q))
print("Latest annual growth: CPI %.1f%%, WPI %.1f%%, real %.1f%%" % tuple(df.loc[latest_q, ["cpi_yoy", "wpi_yoy", "real_yoy"]]))
print("Peak annual CPI: %.1f%% (%s) | WPI growth at that time: %.1f%%" % (df.cpi_yoy.max(), df.cpi_yoy.idxmax(), df.wpi_yoy[df.cpi_yoy.idxmax()]))

## 3. Private vs public sector, and long-run context

In [ ]:
base = df.loc[peak_q]
sector = pd.DataFrame({
    "peak_quarter": str(peak_q),
    "change_peak_to_trough_%": [(df[c].loc[peak_q:].min() / base[c] - 1) * 100 for c in ("real_idx", "real_private", "real_public")],
    "change_peak_to_latest_%": [(df[c].iloc[-1] / base[c] - 1) * 100 for c in ("real_idx", "real_private", "real_public")]},
    index=["All (private + public)", "Private", "Public"]).round(2)
print(sector.to_string()); sector.to_csv("../outputs/sector_real_wages.csv")

# Robustness: 2020Q2 had a temporary CPI dip (COVID-era), so also measure from the pre-COVID quarter 2019Q4
b19 = "2019Q4"
rob = {"base": b19, "to_trough_pct": round((df.real_idx.loc[b19:].min() / df.real_idx[b19] - 1) * 100, 2),
       "trough_quarter": str(df.real_idx.loc[b19:].idxmin()),
       "to_latest_pct": round((df.real_idx.iloc[-1] / df.real_idx[b19] - 1) * 100, 2)}
print("\nRobustness, base 2019Q4:", rob)

def cagr(s, a, b):
    n = (pd.Period(b, "Q") - pd.Period(a, "Q")).n
    return ((s[b] / s[a]) ** (4 / n) - 1) * 100
periods = [("1997Q3", "2012Q4"), ("2012Q4", "2019Q4"), ("2019Q4", "2022Q4"), ("2022Q4", str(latest_q)), ("1997Q3", str(latest_q))]
sub = pd.DataFrame([{"period": f"{a} to {b}", "WPI_%_pa": cagr(df.wpi, a, b), "CPI_%_pa": cagr(df.cpi, a, b),
                     "real_%_pa": cagr(df.real_idx, a, b)} for a, b in periods]).round(2)
print(); print(sub.to_string(index=False)); sub.to_csv("../outputs/subperiod_growth.csv", index=False)

## 4. Charts

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
v = df.loc["2012Q1":]; x = v.index.to_timestamp()
ax[0].plot(x, v.cpi_yoy, label="CPI (annual %)", color="tab:red"); ax[0].plot(x, v.wpi_yoy, label="WPI (annual %)", color="tab:blue")
ax[0].fill_between(x, v.cpi_yoy, v.wpi_yoy, where=v.real_yoy < 0, color="tab:red", alpha=.15, label="Real wages falling")
ax[0].set_ylabel("%"); ax[0].legend(frameon=False, ncol=3); ax[0].set_title("Annual inflation vs wage growth")
ax[1].plot(x, v.real_idx / v.real_idx[b19] * 100, color="k", label="All"); 
ax[1].plot(x, v.real_private / v.real_private[b19] * 100, color="tab:blue", lw=1, label="Private")
ax[1].plot(x, v.real_public / v.real_public[b19] * 100, color="tab:green", lw=1, label="Public")
ax[1].axhline(100, color="grey", ls=":"); ax[1].set_ylabel(f"Real wage index ({b19} = 100)"); ax[1].legend(frameon=False, ncol=3)
ax[1].set_title("Real wages (WPI / CPI), rebased to pre-COVID Dec 2019 quarter")
plt.tight_layout(); plt.savefig("../images/wages_vs_inflation.png"); plt.show()

## 5. Export

In [ ]:
df.round(4).to_csv("../outputs/aligned_series.csv", index_label="quarter")
key = {"data_end": str(latest_q), "peak_quarter": str(peak_q), "trough_quarter": str(trough_q),
       "peak_to_trough_pct": round(drop, 2), "recovered_quarter": str(rec.index[0]) if len(rec) else None,
       "latest_vs_peak_pct": round(vs_peak, 2), "robustness_2019Q4": rob,
       "latest_cpi_yoy": round(df.cpi_yoy.iloc[-1], 2), "latest_wpi_yoy": round(df.wpi_yoy.iloc[-1], 2), "latest_real_yoy": round(df.real_yoy.iloc[-1], 2),
       "peak_cpi_yoy": round(df.cpi_yoy.max(), 2), "peak_cpi_quarter": str(df.cpi_yoy.idxmax()),
       "wpi_yoy_at_peak_cpi": round(df.wpi_yoy[df.cpi_yoy.idxmax()], 2),
       "quarters_real_negative_since_2020": int((df.real_yoy.loc["2020Q1":] < 0).sum()),
       "sector": sector.to_dict(), "series_ids": {"cpi": "A2325846C", "wpi": "A2603609J", "wpi_private": "A2603039T", "wpi_public": "A2603989W"}}
json.dump(key, open("../outputs/key_findings.json", "w"), indent=2); print(json.dumps(key, indent=2))